# Notebook chạy lại baseline

Notebook này gom các script chính để chạy lại pipeline baseline FACT-AUDIT-style trong một chỗ.

## Bước 1: xác định thư mục làm việc

Cell này chuyển về đúng thư mục `fact_audit_reproduction/` trước khi chạy các script.

In [ ]:
from pathlib import Path
import os

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
elif root.name != "fact_audit_reproduction":
    candidate = root / "fact_audit_reproduction"
    if candidate.exists():
        root = candidate

os.chdir(root)
print("Working directory:", Path.cwd())

## Bước 2: hàm chạy lệnh

In [ ]:
import subprocess
import sys

def run_command(args):
    print("$", " ".join(args))
    completed = subprocess.run(args, text=True, capture_output=True)
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    completed.check_returncode()
    return completed

## Bước 3: tạo claim set 30 mẫu

In [ ]:
run_command([sys.executable, "scripts/make_claim_set.py", "--size", "30"])

## Bước 4: chạy smoke test 3 claims

In [ ]:
run_command([sys.executable, "scripts/run_smoke_test.py"])

## Bước 5: chạy baseline đầy đủ 30 claims

Mặc định dùng provider trong `config.yaml`. Nếu muốn ép provider khác, sửa lệnh trong cell này.

In [ ]:
run_command([sys.executable, "scripts/run_baseline.py"])

## Bước 6: chạy demo cache 5 claims

In [ ]:
run_command([sys.executable, "scripts/run_baseline_demo.py", "--use-cache", "--limit", "5"])

## Bước 7: xem nhanh output JSONL/CSV

In [ ]:
import csv
import json
from itertools import islice

baseline_path = Path("outputs/baseline_results.jsonl")
scores_path = Path("outputs/scores.csv")

print("Baseline sample:")
with baseline_path.open(encoding="utf-8") as handle:
    for line in islice(handle, 2):
        row = json.loads(line)
        print({
            "claim_id": row["claim_id"],
            "verdict": row["verdict"],
            "score": row["score"],
            "provider": row["provider"],
        })

print("\nCSV header:")
with scores_path.open(encoding="utf-8") as handle:
    reader = csv.reader(handle)
    print(next(reader))

## Bước 8: chạy với Gemini

Nếu đã có `.env` chứa `GEMINI_API_KEY`, bạn có thể bỏ comment cell dưới để chạy baseline bằng Gemini.

In [ ]:
# run_command([
#     sys.executable,
#     "scripts/run_baseline.py",
#     "--provider",
#     "gemini",
#     "--model",
#     "gemini-2.5-flash",
# ])